## **Code section for retrieving data from Google Drive and resizing.**

In [ ]:
# Connect to google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Resizing images
import os
from PIL import Image

# Global configuration: Define target size for standardizing all images
base_dir = '/content/drive/MyDrive/What horse project/dataset_images'
target_size = (224, 224)

print("Starting image preprocessing in Google Drive...")

# Validate the base directory path
if not os.path.exists(base_dir):
    print(f"Path error: {base_dir} not found.")
else:
    # Loop through each category folder (e.g., horse, zebra)
    for category in os.listdir(base_dir):
        category_path = os.path.join(base_dir, category)

        if os.path.isdir(category_path):
            print(f"Processing category: {category}")

            # Iterate over files in each category
            for filename in os.listdir(category_path):
                # Filter supported image extensions
                if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.jfif')):
                    img_path = os.path.join(category_path, filename)

                    try:
                        with Image.open(img_path) as img:
                            # Standardize image to RGB format (Ensures 3 color channels)
                            img = img.convert('RGB')

                            # Resize using LANCZOS filter for high-quality downsampling
                            img = img.resize(target_size, Image.Resampling.LANCZOS)

                            # Save processed image back to Drive as compressed JPEG
                            img.save(img_path, 'JPEG', quality=90)
                    except Exception as e:
                        # Log issues without interrupting the entire process
                        print(f"Skipping file {filename} due to error: {e}")

    print("--- Image preprocessing completed successfully ---")

In [ ]:
# Create file csv from dataset_images
import os
import pandas as pd
import numpy as np

# Specify the original path you used to store the images
base_dir = '/content/drive/MyDrive/What horse project/dataset_images'

data_list = []

# Loop through and read folder and file names
for category in os.listdir(base_dir):
    category_path = os.path.join(base_dir, category)
    if os.path.isdir(category_path):
        for filename in os.listdir(category_path):
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                # Define basic characteristics based on animal species
                has_horn = 1 if category in ['unicorn', 'alicorn'] else 0
                has_wings = 1 if category in ['pegasus', 'alicorn'] else 0
                is_striped = 1 if category == 'zebra' else 0

                # Add information to a list
                data_list.append({
                    'image_filename': filename,
                    'category': category,
                    'has_horn': has_horn,
                    'has_wings': has_wings,
                    'is_striped': is_striped
                })

# Convert to a table
df = pd.DataFrame(data_list)

# Randomly insert Missing Values ​​(NaN)
for col in ['has_horn', 'has_wings']:
    # Randomly select 25 rows as indexes to make the data empty
    missing_idx = df.sample(n=25, random_state=42).index
    df.loc[missing_idx, col] = np.nan

# Save as a CSV file
csv_path = '/content/drive/MyDrive/What horse project/animal_features.csv'
df.to_csv(csv_path, index=False)

print(f"CSV file created successfully! Total number of rows: {len(df)}")
print(f"Saved at: {csv_path}")

In [ ]:
# Count images
import os

# Define the root directory of the processed dataset
base_dir = '/content/drive/MyDrive/What horse project/dataset_images'

# Iterate through each folder to verify the results
for category in os.listdir(base_dir):
    path = os.path.join(base_dir, category)

    # Ensure we are only counting files inside sub-directories
    if os.path.isdir(path):
        # Count only files (excluding subfolders) within the category
        num_files = len([f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))])

        # Display the summary of class distribution
        print(f"Category {category}: {num_files} images were found.")